# Load Packages

In [1]:
import os
import sys
from pathlib import Path
import numpy as np
import openpyxl
import pandas as pd
import torch
from transformers import pipeline


In [2]:
# Custom functions
from custom_functions import (
    get_device,
    get_pipeline_device_id,
    normalize_label,
    analyze_long_text,
    analyze_sentiment,
    clean_text,
)


# Parameters

In [3]:
# Output directory — all plots and tables land here
output_dir = os.path.join("..", "outputs")
os.makedirs(output_dir, exist_ok=True)

# Data directory — all data files land here
data_raw_dir = os.path.join("..", "data", "raw")
data_processed_dir = os.path.join("..", "data", "processed")

# Seed for reproducibility
np.random.seed(42)

# Load and Pre-Process Data

In [4]:
# Load dataset
toaster_file_path = os.path.join(data_raw_dir, "Param_Toaster Data_2018_23.xlsx")
toaster_df = pd.read_excel(toaster_file_path)

# Data preview
toaster_df.head()

,ASIN,P_TITLE,OP,DP,SP,FS,PRA_4.5,P_RTG,RTG_P_NO,SELLER_LINK,...,RV_DT,VP,HLP_VT,IMG_PRST,TTL_RV,RVS_L,RV_TRANS,SUBJ,SRVS,CP_RVS
0,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
1,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
2,B009GQ034C,Cuisinart CPT-122 Compact Plastic 2-Slice Toas...,55.462963,0.460000,29.95,1,0,4.3,27270,https://www.amazon.com/stores/Cuisinart/page/9...,...,2018-01-01,1,.,0,6628,24,"Very happy. Thanks, Tom.",0.6,positive,0.765
3,B0744M3SB4,Nostalgia TCS2 Grilled Cheese Toaster with Eas...,209.988477,0.786655,44.8,1,0,4.1,4156,https://www.amazon.com/stores/Nostalgia/page/B...,...,2018-01-01,1,12,1,863,413,"I bought this for me husband for Christmas, af...",0.508333,positive,0.4754
4,B07H81RZ9Q,Hamilton Beach 2 Slice Extra Wide Slot Toaster...,58.890000,0.000000,58.89,1,0,4.2,9529,https://www.amazon.com/stores/HamiltonBeach/pa...,...,2018-01-01,1,1,0,4979,106,"Worked OK, never above average. Died one year ...",0.45,negative,-0.6908


In [5]:
toaster_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 85262 entries, 0 to 85261
Data columns (total 30 columns):
 #   Column       Non-Null Count  Dtype         
---  ------       --------------  -----         
 0   ASIN         85262 non-null  str           
 1   P_TITLE      85262 non-null  str           
 2   OP           67023 non-null  float64       
 3   DP           85262 non-null  float64       
 4   SP           85262 non-null  object        
 5   FS           85262 non-null  object        
 6   PRA_4.5      85262 non-null  int64         
 7   P_RTG        85262 non-null  object        
 8   RTG_P_NO     85262 non-null  object        
 9   SELLER_LINK  85262 non-null  str           
 10  IMAGE_URL    85262 non-null  str           
 11  P_URL        85262 non-null  str           
 12  RV_URL       85262 non-null  str           
 13  PRFL_IMG     85262 non-null  str           
 14  PRFL_URL     85262 non-null  str           
 15  RV_TTL       85259 non-null  str           
 16  RVS          85

In [6]:
# Size before deduplication
print(f"Dataset size before deduplication: {toaster_df.shape[0]} rows")

# Format RV_DT as datetime
toaster_df["RV_DT"] = pd.to_datetime(toaster_df["RV_DT"])

# Remove duplicate rows by keeping the first date (RV_DT) for each unique combination of reviewer (RVR) and product (ASIN)
toaster_df = (
    toaster_df.sort_values(["RVR", "ASIN", "RV_DT"], ascending=[True, True, False])
    .drop_duplicates(subset=["RVR", "ASIN"], keep="first")
    .reset_index(drop=True)
)

# Size after deduplication
print(f"Dataset size after deduplication: {toaster_df.shape[0]} rows")

Dataset size before deduplication: 85262 rows
Dataset size after deduplication: 62014 rows


## Save Deduplicated Data

In [7]:
toaster_df.to_csv(os.path.join(data_processed_dir, "toaster_dedup.csv"), index=False)

# Sentiment Analysis

## Device Detection
This will help detect if there exist a GPU on the device.

In [8]:
device = get_device()
device_id = get_pipeline_device_id(device)

Using Apple MPS (Apple M2 Max)


## Load Sentiment Pipeline

In [9]:
# Load Sentiment Pipeline
# Model: fine-tuned RoBERTa, handles diverse product review language well

# model_name = "nlptown/bert-base-multilingual-uncased-sentiment"
# model_name = "distilbert-base-uncased-finetuned-sst-2-english"
# model_name = "cardiffnlp/twitter-roberta-base-sentiment"
# model_name = "microsoft/deberta-v3-base"

model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model=model_name,
    tokenizer=model_name,
    device=device_id,
    truncation=True,        # Handles long reviews (truncates to 512 tokens)
    max_length=512,
    batch_size=32,          # Process in batches for speed
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [10]:
# Label Mapping
LABEL_MAP = {
    # Standard text labels
    "positive": "positive",
    "negative": "negative",
    "neutral":  "neutral",
    # Numeric labels (verify order on the model's HuggingFace card!)
    "label_0":  "negative",
    "label_1":  "neutral",
    "label_2":  "positive",
}

## Run & Display 

In [11]:
# take a sample to speed up testing — remove or increase for full analysis
# toaster_df = toaster_df.sample(50, random_state=42).copy()

# Analyze sentiment and add results to the DataFrame
toaster_df = analyze_sentiment(
    toaster_df,
    sentiment_pipeline=sentiment_pipeline,
    text_col="RV_TRANS",
    label_map=LABEL_MAP,
)

# Rename BERT output columns for clarity
toaster_df = toaster_df.rename(
    columns={
        "SENTIMENT": "BERT_SENT",
        "SENTIMENT_SCORE": "BERT_SCORE",
    }
)


Analyzing 61395 texts (skipping 619 empty/invalid)...

Short texts (direct batch): 61305
Long texts (chunking + voting): 90



Long texts: 100%|██████████| 90/90 [00:04<00:00, 19.56it/s]


Done. 90 text(s) used chunking + voting.


## Results

In [12]:
# Review the results
cols = ["RV_TRANS", "SRVS", "CP_RVS", "BERT_SENT", "BERT_SCORE"]
pd.set_option("display.max_colwidth", 120)
display(toaster_df[cols].head(10))


,RV_TRANS,SRVS,CP_RVS,BERT_SENT,BERT_SCORE
0,Love making 4 slices at a time.,positive,0.6369,positive,0.967
1,Great item to use if you love grilled cheese but you have to use thin sliced bread only downfall,positive,0.8519,positive,0.8974
2,I had high hopes for this toaster - but it takes two cycles to brown properly.,positive,0.4404,negative,0.7371
3,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,positive,0.9914,neutral,0.6821
4,.,.,.,<NA>,<NA>
5,"This oven works very well, and does everything it's supposed to do. I'm very happy with it!",positive,0.5719,positive,0.9867
6,"I received this toaster as a gift. Really great product. Does a great job with bagels and bread, but also does a goo...",positive,0.9544,positive,0.9694
7,"Love this toaster, it has super wide slots that accommodates larger width Italian breads, plus toast a bagel easily....",positive,0.9638,positive,0.9788
8,Clear view to toasting indeed. Have not had issues at all with toasted items popping up to high as I have heard and ...,positive,0.7717,positive,0.7818
9,Only toasts one side of the sandwich. Slots are way to thin for your regular every day white bread. Can only put one...,negative,-0.2023,negative,0.9315


In [13]:
# Check distribution of sentiment labels and review the long reviews that were chunked
print("\nSentiment Distribution:")
display(toaster_df["BERT_SENT"].value_counts())

print("\nLong reviews that used chunking:")
display(toaster_df[toaster_df["CHUNKED"]][["RV_TRANS", "BERT_SENT", "BERT_SCORE"]])


Sentiment Distribution:


BERT_SENT
positive    37406
negative    17124
neutral      6865
Name: count, dtype: int64


Long reviews that used chunking:


,RV_TRANS,BERT_SENT,BERT_SCORE
3,BE AWARE: Read all the instructions included as this warning exists there: the first time (and maybe the 2nd & 3rd t...,neutral,0.6821
196,[UPDATED at bottom of review]\n-------------------------------------\nWe are reserving final judgment on this until ...,neutral,0.8554
741,"I just looked at the date we bought this, it was about 15 months ago. Within the first few weeks, we realize the so...",negative,0.7893
938,"Over the years, before today, we had purchased three of the Breville 830XL toasters and our daughter has (on our rec...",neutral,0.6841
1159,"Like so many others, I've recently started baking my own sourdough bread. I like my bread toasted, and didn't know t...",positive,0.8307
...,...,...,...
56008,"I am a fussy toast person, I like it just so.\nI had a nice inexpensive toaster for years and then it started to bre...",positive,0.5268
58835,"I own the Black & Decker TO1303SBD toaster oven, which is a slightly older model. I bought it in April 2020, and so...",positive,0.697
59673,"I'd give it 5 stars but for a toaster this expensive, you'd think the crumb tray would be well built and sized prope...",negative,0.6492
60085,"I wouldn't say that I'm a toast connoisseur, but I appreciate nice toast when I have it. Over my life I've probably ...",positive,0.8089


## Save Results to File

In [14]:
toaster_df.to_csv(os.path.join(data_processed_dir, "toaster_bert.csv"), index=False)

# Compare Results from BERT with VARDER

In [15]:
# Compare SRVS and BERT_SENT
compare_df = toaster_df[["SRVS", "BERT_SENT", "RV_TRANS"]].copy()

# Normalize text labels
compare_df["SRVS_clean"] = (
    compare_df["SRVS"]
    .astype("string")
    .str.strip()
    .str.lower()
)

compare_df["BERT_SENT_clean"] = (
    compare_df["BERT_SENT"]
    .astype("string")
    .str.strip()
    .str.lower()
)

# Keep only rows where both columns are available
compare_valid = compare_df.dropna(subset=["SRVS_clean", "BERT_SENT_clean"]).copy()

# Check match
compare_valid["MATCH"] = (
    compare_valid["SRVS_clean"] == compare_valid["BERT_SENT_clean"]
)

# Agreement rate
agreement_rate = compare_valid["MATCH"].mean()

print(f"Agreement rate: {agreement_rate:.2%}")
print(f"Compared rows: {len(compare_valid)}")

# Cross-tab comparison
pd.crosstab(
    compare_valid["SRVS_clean"],
    compare_valid["BERT_SENT_clean"],
    margins=True
)


Agreement rate: 72.77%
Compared rows: 61395


BERT_SENT_clean,negative,neutral,positive,All
SRVS_clean,,,,
negative,6561,835,762,8158
neutral,2541,2847,1377,6765
positive,8022,3183,35267,46472
All,17124,6865,37406,61395
